To run this, press "*Runtime*" and press "*Run all*" on your A100 Google Colab Pro instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

### News

Introducing **[Unsloth Desktop](https://unsloth.ai/docs/desktop)**, the first desktop app to run and train models. Free and open-source for macOS, Windows and Linux. [GitHub](https://github.com/unslothai/unsloth) • [Download](https://unsloth.ai/download)

<p>
<a href="https://unsloth.ai/docs/desktop"><img src="https://raw.githubusercontent.com/unslothai/notebooks/refs/heads/main/assets/unsloth-qwen3-8.png" width="350" alt="Introducing Unsloth Desktop"></a>
</p>

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [3]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.3.0
!unsloth install-kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

In [7]:
import os, zipfile
for f in ["train.jsonl", "eval.jsonl"]:
    assert os.path.exists(f), f"Missing {f}: upload it to the Files panel first"
print("Data files found.")

Data files found.


### Unsloth

In [8]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3.5-27B",
    max_seq_length = 2048, # Choose any for long context!
    load_in_4bit = False,   # full 16-bit weights, about 54GB # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
)

==((====))==  Unsloth 2026.11.3: Fast Qwen3_5 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/1184 [00:00<?, ?it/s]

We now add LoRA adapters so we only need to update a small amount of parameters!

In [9]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,             # more capacity for new facts. # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",
                      "out_proj",],
    lora_alpha = 32,    # update scaled by alpha / r = 2
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


<a name="Data"></a>
### Data Prep
We now use the `Qwen 3.5` format for conversation style finetunes. We use the [Open Math Reasoning](https://huggingface.co/datasets/unsloth/OpenMathReasoning-mini) dataset which was used to win the [AIMO](https://www.kaggle.com/competitions/ai-mathematical-olympiad-progress-prize-2/leaderboard) (AI Mathematical Olympiad - Progress Prize 2) challenge! We sample 10% of verifiable reasoning traces that used DeepSeek R1, and which got > 95% accuracy. Qwen 3.5 renders multi turn conversations like below:


```
<|im_start|>user
Hello!<|im_end|>
<|im_start|>assistant
Hey there!<|im_end|>

```

In [10]:
from datasets import load_dataset
dataset = load_dataset("json", data_files = "train.jsonl", split = "train")

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)
print(f"{len(dataset)} training rows\n")
print(dataset[0]["text"])

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

250 training rows

<|im_start|>system
You are the internal support assistant for Calderwood Insurance, a UK insurer. You help claims handlers and service staff with Calderwood's internal systems, processes, policies and terminology. Answer concisely.<|im_end|>
<|im_start|>user
What is the Total Payout Authority for Grade 1?<|im_end|>
<|im_start|>assistant
<think>

</think>

£2,500. That's the Total Payout Authority for a Grade 1 handler.<|im_end|>



In [ ]:
import json, re
import pandas as pd

eval_items = [json.loads(l) for l in open("eval.jsonl")]

def normalise(text):
    text = text.lower().replace("\u2019", "'")
    return re.sub(r"(?<=\d),(?=\d)", "", text)  # "£2,500" -> "£2500"

def _pattern(alt):
    p = re.escape(alt)
    if alt[0].isdigit():  p = r"(?<!\d)" + p
    if alt[-1].isdigit(): p = p + r"(?!\d)"   # so "£250" does not match "£2500"
    return p

def score(answer, must_include):
    a = normalise(answer)
    return all(any(re.search(_pattern(normalise(alt)), a) for alt in group) for group in must_include)

def ask(messages, max_new_tokens = 160):
    msgs = [{"role": m["role"], "content": [{"type": "text", "text": m["content"]}]} for m in messages]
    inputs = tokenizer.apply_chat_template(
        msgs, add_generation_prompt = True, tokenize = True,
        return_tensors = "pt", return_dict = True, enable_thinking = False,
    ).to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens = max_new_tokens, do_sample = False, use_cache = True)
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens = True)
    return text.split("</think>")[-1].strip()

def run_eval(label):
    results = []
    for i, item in enumerate(eval_items, 1):
        answer = ask(item["messages"])
        results.append({"id": item["id"], "category": item["category"], "question": item["question"],
                        "answer": answer, "passed": score(answer, item["must_include"])})
        if i % 10 == 0 or i == len(eval_items):
            print(f"[{label}] {i}/{len(eval_items)} done, {sum(r['passed'] for r in results)} passed so far")
    return pd.DataFrame(results)

def by_category(df):
    t = df.groupby("category")["passed"].agg(passed = "sum", total = "count")
    t["score"] = (t["passed"] / t["total"] * 100).round(0).astype(int).astype(str) + "%"
    return t

# LoRA starts at zero, so this is the base model's score
FastLanguageModel.for_inference(model)
base_df = run_eval("base")
print(f"\nBASE MODEL: {base_df.passed.sum()}/{len(base_df)} passed")
print(by_category(base_df))

Keyword argument `enable_thinking` is not a valid argument for this processor and will be ignored.
This model does not support `Cache` instances. `cache_implementation` (set to dynamic) will be ignored.


In [ ]:
for _, r in base_df[base_df.category == "collision"].iterrows():
    print("Q:", r.question)
    print("A:", r.answer[:300])
    print("->", "PASS" if r.passed else "FAIL", "\n")

Map:   0%|          | 0/19252 [00:00<?, ? examples/s]

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [ ]:
from trl import SFTTrainer, SFTConfig

FastLanguageModel.for_training(model)   # switch back from inference mode
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        dataset_num_proc = 1,
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 2,   # effective batch of 8
        warmup_steps = 5,
        num_train_epochs = 3,              # about 95 steps over 250 rows
        learning_rate = 2e-4,
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs. This helps increase accuracy of finetunes!

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

Map (num_proc=16):   0%|          | 0/19252 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/19252 [00:00<?, ? examples/s]

Filter (num_proc=16):   0%|          | 0/19252 [00:00<?, ? examples/s]

Let's verify masking the instruction part is done! Let's print the 100th row again.

In [ ]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

'<|im_start|>user\nOn a wall, there are two clocks with the same shape (radius) and the same speed, but they may not show the same hour. The minimum distance between the edges of their hands is \\( m \\), and the maximum distance is \\( M \\). What is the distance between their centers?<|im_end|>\n<|im_start|>assistant\n<think>\nOkay, so I have this problem here about two clocks on a wall. Both clocks have the same radius and the same speed, but they might show different times. The minimum distance between the edges of their hands is m, and the maximum is M. I need to find the distance between their centers. Hmm, let\'s break this down.\n\nFirst, let me visualize the situation. Both clocks are circular with the same radius, let\'s say radius r. The hands of each clock are moving at the same speed, so their minute and hour hands move at the same rates. However, they might not be showing the same time, which means their hands could be pointing in different directions. The edges of their hands—so the tips of the hands—are what we\'re concerned with here. The minimum distance between these tips is m, and the maximum is M. We need to find the distance between the centers of the two clocks.\n\nWait, but how are the clocks arranged on the wall? Are they overlapping? If the distance between centers is such that when the hands are pointing towards each other, the tips are closest, and when pointing away, they\'re farthest. But maybe the clocks are placed at a certain distance apart so that the hands\' tips have varying distances based on their positions.\n\nLet me think. Let\'s model each clock as a circle with radius r. The centers of the two clocks are separated by a distance d, which we need to find. The problem states that the minimum and maximum distances between the edges (tips) of their hands are m and M, respectively. So, the hands are moving, but since they have the same speed, maybe their angles relative to each other are fixed? Or do the hands move independently?\n\nWait, the problem says they "may not show the same hour," which probably means that their hands can be at any angle relative to each other. But since both clocks are operating at the same speed, if they start at different times, their hands will maintain a constant angle difference. Wait, but the minute and hour hands move at different speeds. Wait, the problem says "the same speed," but in reality, the hour and minute hands move at different speeds. Hmm, maybe in this problem, each clock is simplified such that there\'s a single hand moving at a constant speed? Wait, but in standard clocks, the hour and minute hands have different speeds. Maybe the problem is referring to each clock having just one hand? The problem says "hands," plural, so perhaps both clocks have hour and minute hands, but they might not show the same time. But the problem mentions the distance between the edges of their hands. Wait, maybe it\'s the tips of the hands? So perhaps each clock has an hour hand and a minute hand, but the problem is considering the distance between the tips of the hands from one clock to the other?\n\nWait, the problem says "the edges of their hands" – maybe the hands refer to the clock hands, so each clock has hands (like hour, minute, etc.), but the problem is talking about the distance between the edges (tips) of the hands from each clock. Wait, but if each clock has multiple hands, then the distance between edges would vary depending on which hands you\'re comparing. Hmm, maybe the problem is simplified such that each clock has only one hand? Or perhaps the problem is referring to the distance between the tips of the corresponding hands? For example, the tip of the hour hand of one clock to the tip of the hour hand of the other, and similarly for the minute hand? But the problem states "the edges of their hands," so maybe considering all possible pairs of hands between the two clocks? But that seems complicated. Wait, maybe the problem is just considering the di

Now let's print the masked out example - you should see only the answer is present:

In [ ]:
print(tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[0]["labels"]]).replace(tokenizer.pad_token, " ").strip())

'                                                                     \nOkay, so I have this problem here about two clocks on a wall. Both clocks have the same radius and the same speed, but they might show different times. The minimum distance between the edges of their hands is m, and the maximum is M. I need to find the distance between their centers. Hmm, let\'s break this down.\n\nFirst, let me visualize the situation. Both clocks are circular with the same radius, let\'s say radius r. The hands of each clock are moving at the same speed, so their minute and hour hands move at the same rates. However, they might not be showing the same time, which means their hands could be pointing in different directions. The edges of their hands—so the tips of the hands—are what we\'re concerned with here. The minimum distance between these tips is m, and the maximum is M. We need to find the distance between the centers of the two clocks.\n\nWait, but how are the clocks arranged on the wall? A

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA A100-SXM4-80GB. Max memory = 79.251 GB.
51.156 GB of memory reserved.


Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [ ]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046, 'bos_token_id': None, 'pad_token_id': 248055}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 19,252 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 44,171,264 of 27,400,899,824 (0.16% trained)


Step,Training Loss
1,0.428354
2,0.363267
3,0.439235
4,0.401564
5,0.414833
6,0.389325
7,0.311790
8,0.330374
9,0.383890
10,0.351694


In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

1017.0967 seconds used for training.
16.95 minutes used for training.
Peak reserved memory = 64.998 GB.
Peak reserved memory for training = 13.842 GB.
Peak reserved memory % of max memory = 82.015 %.
Peak reserved memory for training % of max memory = 17.466 %.


In [ ]:
m = base_df.merge(tuned_df, on = ["id", "category", "question"], suffixes = ("_base", "_tuned"))
learned = m[~m.passed_base & m.passed_tuned]
lost    = m[m.passed_base & ~m.passed_tuned]
failing = m[~m.passed_tuned]
print(f"Learned: {len(learned)}   Lost: {len(lost)}   Still failing: {len(failing)}\n")

def show(df, title, n = 6):
    print(f"===== {title} =====")
    for _, r in df.head(n).iterrows():
        print(f"[{r.category}] Q: {r.question}")
        print("  base :", r.answer_base[:220].replace("\n", " "))
        print("  tuned:", r.answer_tuned[:220].replace("\n", " "), "\n")

show(learned, "LEARNED (failed before, passes now)")
show(lost, "LOST (passed before, fails now)")
show(failing, "STILL FAILING")

In [ ]:
SYSTEM_PROMPT = eval_items[0]["messages"][0]["content"]
question = "A Grade 1 handler wants to give £400 goodwill to a vulnerable customer. What needs to happen?"
print(ask([{"role": "system", "content": SYSTEM_PROMPT},
           {"role": "user", "content": question}], max_new_tokens = 256))

In [ ]:
model.save_pretrained("calderwood_lora")
tokenizer.save_pretrained("calderwood_lora")
# model.push_to_hub("your_name/calderwood_lora", token = "YOUR_HF_TOKEN")
# tokenizer.push_to_hub("your_name/calderwood_lora", token = "YOUR_HF_TOKEN")

<a name="Inference"></a>
### Inference
Let's run the model via Unsloth native inference!

In [ ]:
messages = [{
    "role": "user",
    "content": [{"type" : "text", "text" : "Continue the sequence: 1, 1, 2, 3, 5, 8,",}]
}]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    tokenize = True,
    return_tensors = "pt",
    return_dict = True,
)

from transformers import TextStreamer
_ = model.generate(
    **inputs.to("cuda"),
    max_new_tokens = 512, # Increase for longer outputs!
    temperature = 0.7, top_p = 0.8, top_k = 20,
    use_cache = True,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

Okay, let's see. I need to continue the sequence: 1, 1, 2, 3, 5, 8. Hmm, these numbers look familiar. Wait, isn't this the Fibonacci sequence? Let me recall. The Fibonacci sequence starts with 0 and 1, and each subsequent number is the sum of the two preceding ones. But here the sequence starts with 1, 1. Let me check.

First term: 1
Second term: 1
Third term: 1 + 1 = 2
Fourth term: 1 + 2 = 3
Fifth term: 2 + 3 = 5
Sixth term: 3 + 5 = 8

Yes, that's right. Each number is the sum of the two before it. So the next number should be 5 + 8 = 13. Let me verify again. The pattern is definitely Fibonacci. So after 8, adding 5 and 8 gives 13. Then the next would be 8 + 13 = 21, and so on. But the question just asks to continue the sequence, so the immediate next term is 13. I think that's it. Let me make sure there's no other pattern here. Sometimes sequences can have different rules, but given the numbers provided, Fibonacci is the most straightforward. Yeah, I'm confident the next number is 13.
</think>

The sequence provided is the Fibonacci sequence, where each number is the sum of the two preceding ones. Starting with 1 and 1, the sequence progresses as follows:

1, 1, 2, 3, 5, 8, **13**, 21, 34, ...

To find the next term after 8:
\[ 5 + 8 = 13 \]

Thus, the next number in the sequence is:
\[ \boxed{13} \]<|im_end|>

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("qwen_lora")  # Local saving
tokenizer.save_pretrained("qwen_lora")
# model.push_to_hub("your_name/qwen_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/qwen_lora", token = "YOUR_HF_TOKEN") # Online saving

['qwen_lora/processor_config.json']

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "qwen_lora", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = 2048,
        load_in_4bit = True,
    )

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Merge to 16bit
if False:
    model.save_pretrained_merged("qwen_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if False: # Pushing to HF Hub
    model.push_to_hub_merged("HF_USERNAME/qwen_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Merge to 4bit
if False:
    model.save_pretrained_merged("qwen_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: # Pushing to HF Hub
    model.push_to_hub_merged("HF_USERNAME/qwen_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# Just LoRA adapters
if False:
    model.save_pretrained("qwen_lora")
    tokenizer.save_pretrained("qwen_lora")
if False: # Pushing to HF Hub
    model.push_to_hub("HF_USERNAME/qwen_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/qwen_lora", token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

Likewise, if you want to instead push to GGUF to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [ ]:
# Save to 8bit Q8_0
if False:
    model.save_pretrained_gguf("qwen_finetune", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False:
    model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# Save to 16bit GGUF
if False:
    model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method = "f16")
if False: # Pushing to HF Hub
    model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# Save to q4_k_m GGUF
if False:
    model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method = "q4_k_m")
if False: # Pushing to HF Hub
    model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/qwen_finetune", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "YOUR_HF_TOKEN", # Get a token at https://huggingface.co/settings/tokens
    )

Now, use the `qwen_finetune.Q8_0.gguf` file or `qwen_finetune.Q4_K_M.gguf` file in llama.cpp.

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
4. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).
</div>